In [ ]:
"""
=============================================================================
DETERMINISTIC FINITE AUTOMATON (DFA) FORMAL DEFINITION (5-Tuple):
    M = (Q, Σ, δ, q0, F)
Where:
    1. Q   : Finite set of states                      -> list/set of state names
    2. Σ   : Finite set of input symbols (Alphabet)    -> set of valid symbols
    3. δ   : Transition function: Q × Σ -> Q           -> transitions[(state, symbol)] = next_state
    4. q0  : Initial / Start state (q0 ∈ Q)           -> start_state
    5. F   : Set of Final / Accept states (F ⊆ Q)     -> accept_states
=============================================================================
"""
class UniversalDFA:

  def __init__(self, states, alphabet, transitions, start_state, accept_states):
    self.states = list(states)
    self.alphabet = set(alphabet)
    self.transitions = transitions  # {(state, symbol): next_state}
    self.start_state = start_state
    self.accept_states = set(accept_states)

  def simulate(self, input_string):
    current_state = self.start_state
    path = [(current_state, None)]  # (state, symbol_consumed)

    for char in input_string:
      if char not in self.alphabet:
        return False, path, f"Invalid symbol '{char}' not in alphabet."
      transition_key = (current_state, char)
      if transition_key not in self.transitions:
        return False, path, f"No transition defined for ({current_state}, {char})."

      current_state = self.transitions[transition_key]
      path.append((current_state, char))

    is_accepted = current_state in self.accept_states
    return is_accepted, path, None

  def print_diagram(self):
    print("\n" + "=" * 45)
    print("           DFA TRANSITION TABLE")
    print("=" * 45)
    sorted_alpha = sorted(list(self.alphabet))
    header = f"{'State':^12} | " + " | ".join(
        [f"Input '{a}':^10" for a in sorted_alpha]
    )
    print(header)
    print("-" * len(header))

    for s in self.states:
      prefix = "-> " if s == self.start_state else "   "
      suffix = " *" if s in self.accept_states else "  "
      row_state = f"{prefix}{s}{suffix}"

      row_vals = []
      for a in sorted_alpha:
        dest = self.transitions.get((s, a), "DEAD")
        row_vals.append(f"{dest:^10}")

      print(f"{row_state:<12} | " + " | ".join(row_vals))
    print("=" * 45)
    print("Legend: '->' = Start State, '*' = Accept State\n")

  def draw_execution_trace(self, path, is_accepted):
    print("\n" + "-" * 50)
    print("             VISUAL STATE FLOW")
    print("-" * 50)

    for i, (state, symbol) in enumerate(path):
      is_accept = " [ACCEPT]" if state in self.accept_states else ""
      is_start = " (START)" if i == 0 else ""

      if i == 0:
        print(f"  +-------------------------+")
        print(f"  | Current State: {state:<7}{is_start:<8}|")
        print(f"  +-------------------------+")
      else:
        print(f"             |")
        print(f"        Read: [{symbol}]")
        print(f"             v")
        print(f"  +-------------------------+")
        print(f"  | State: {state:<9}{is_accept:<8} |")
        print(f"  +-------------------------+")

    status = (
        " ACCEPTED (Valid Member)" if is_accepted else " REJECTED (Invalid)"
    )
    print("-" * 50)
    print(f" Final Verdict: {status}")
    print("-" * 50 + "\n")


def run():
  print("=== Universal DFA Simulator (No External Libraries) ===")
  alphabet = input("Enter alphabet symbols (e.g. 0 1): ").split()
  states = input("Enter state names (e.g. q0 q1): ").split()
  start_state = input("Enter start state: ").strip()
  accept_states = input("Enter accept state(s): ").split()

  print("\nEnter transitions:")
  transitions = {}
  for s in states:
    for a in alphabet:
      dest = input(f"  δ({s}, {a}) -> ").strip()
      transitions[(s, a)] = dest

  dfa = UniversalDFA(states, alphabet, transitions, start_state, accept_states)
  dfa.print_diagram()

  while True:
    test_str = input(
        "Enter string to test (or 'exit' to quit): "
    ).strip()
    if test_str.lower() == "exit":
      break

    accepted, path, error = dfa.simulate(test_str)
    if error:
      print(f"\n[!] Error: {error}\n")
    else:
      dfa.draw_execution_trace(path, accepted)


if __name__ == "__main__":
  run()

=== Universal DFA Simulator (No External Libraries) ===
Enter alphabet symbols (e.g. 0 1): 0 1
Enter state names (e.g. q0 q1): q0 q1 q2
Enter start state: q0
Enter accept state(s): q2

Enter transitions:
  δ(q0, 0) -> q1
  δ(q0, 1) -> q0
  δ(q1, 0) -> q1
  δ(q1, 1) -> q2
  δ(q2, 0) -> q2
  δ(q2, 1) -> q2

           DFA TRANSITION TABLE
   State     | Input '0':^10 | Input '1':^10
--------------------------------------------
-> q0        |     q1     |     q0    
   q1        |     q1     |     q2    
   q2 *      |     q2     |     q2    
Legend: '->' = Start State, '*' = Accept State

Enter string to test (or 'exit' to quit): 11010

--------------------------------------------------
             VISUAL STATE FLOW
--------------------------------------------------
  +-------------------------+
  | Current State: q0      (START)|
  +-------------------------+
             |
        Read: [1]
             v
  +-------------------------+
  | State: q0                |
  +----------------

In [ ]:
"""
=============================================================================
NONDETERMINISTIC FINITE AUTOMATON (NFA) FORMAL DEFINITION (5-Tuple):
    M = (Q, Σ, δ, q0, F)
Where:
    1. Q   : Finite set of states                      -> set of state names
    2. Σ   : Finite set of input symbols (Alphabet)    -> set of symbols (excluding 'e')
    3. δ   : Transition function: Q × (Σ ∪ {ε}) -> P(Q) -> transitions[(state, symbol)] = {next_states}
    4. q0  : Initial / Start state (q0 ∈ Q)           -> start_state
    5. F   : Set of Final / Accept states (F ⊆ Q)     -> accept_states
=============================================================================
"""

from collections import defaultdict

class UniversalNFA:
    def __init__(self, states, alphabet, transitions, start_state, accept_states):
        self.states = set(states)
        self.alphabet = set(alphabet)
        self.transitions = transitions  # {(state, symbol): {next_states}}
        self.start_state = start_state
        self.accept_states = set(accept_states)

    def epsilon_closure(self, states):
        """Finds all states reachable via epsilon (e) transitions."""
        stack = list(states)
        closure = set(states)
        while stack:
            state = stack.pop()
            next_states = self.transitions.get((state, 'e'), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return closure

    def simulate(self, input_string):
        # Start state with epsilon closure
        current_states = self.epsilon_closure({self.start_state})
        history = [(current_states, None)]

        for char in input_string:
            if char not in self.alphabet:
                return False, history, f"Invalid symbol '{char}' not in alphabet."

            next_states = set()
            for state in current_states:
                # Direct transitions on symbol
                direct = self.transitions.get((state, char), set())
                # Follow epsilon transitions from destinations
                next_states.update(self.epsilon_closure(direct))

            current_states = next_states
            history.append((current_states, char))
            if not current_states:
                break  # All branches died

        is_accepted = bool(current_states & self.accept_states)
        return is_accepted, history, None

    def print_diagram(self):
        print("\n" + "=" * 55)
        print("                 NFA TRANSITION TABLE")
        print("=" * 55)
        symbols = sorted(list(self.alphabet)) + (['e'] if any(k[1] == 'e' for k in self.transitions) else [])
        header = f"{'State':^12} | " + " | ".join([f"'{s}':^8" for s in symbols])
        print(header)
        print("-" * len(header))

        for s in sorted(self.states):
            prefix = "-> " if s == self.start_state else "   "
            suffix = " *" if s in self.accept_states else "  "
            row_state = f"{prefix}{s}{suffix}"

            row_vals = []
            for sym in symbols:
                dest_set = self.transitions.get((s, sym), set())
                dest_str = "{" + ",".join(sorted(dest_set)) + "}" if dest_set else "∅"
                row_vals.append(f"{dest_str:^8}")

            print(f"{row_state:<12} | " + " | ".join(row_vals))
        print("=" * 55)
        print("Legend: '->' = Start State, '*' = Accept State, 'e' = Epsilon (ε)\n")

    def draw_execution_trace(self, history, is_accepted):
        print("\n" + "-" * 55)
        print("           PARALLEL BRANCH EXECUTION FLOW")
        print("-" * 55)

        for i, (states, symbol) in enumerate(history):
            states_str = "{" + ", ".join(sorted(states)) + "}" if states else "{ DEAD / NO PATH }"
            accept_hits = states & self.accept_states
            accept_badge = f" [ACCEPTS: {', '.join(accept_hits)}]" if accept_hits else ""

            if i == 0:
                print(f"  +---------------------------------------------+")
                print(f"  | Active States: {states_str:<29}|")
                print(f"  +---------------------------------------------+")
            else:
                print(f"                     |")
                print(f"                Read: [{symbol}]")
                print(f"                     v")
                print(f"  +---------------------------------------------+")
                print(f"  | Active States: {states_str:<29}{accept_badge}")
                print(f"  +---------------------------------------------+")

        status = "ACCEPTED (At least one active path reached accept state)" if is_accepted else "REJECTED (No path reached accept state)"
        print("-" * 55)
        print(f" Final Verdict: {status}")
        print("-" * 55 + "\n")


def run():
    print("=== Universal NFA Simulator ===")
    alphabet = input("Enter alphabet symbols (e.g. 0 1): ").split()
    states = input("Enter state names (e.g. q0 q1 q2): ").split()
    start_state = input("Enter start state: ").strip()
    accept_states = input("Enter accept state(s): ").split()

    print("\nEnter transitions. Type destination states separated by space (or press Enter if none):")
    transitions = defaultdict(set)

    symbols_to_ask = alphabet + ['e']  # 'e' for epsilon
    for s in states:
        for sym in symbols_to_ask:
            dest = input(f"  δ({s}, {'ε' if sym == 'e' else sym}) -> ").strip()
            if dest:
                transitions[(s, sym)] = set(dest.split())

    nfa = UniversalNFA(states, alphabet, transitions, start_state, accept_states)
    nfa.print_diagram()

    while True:
        test_str = input("Enter string to test (or 'exit' to quit): ").strip()
        if test_str.lower() == "exit":
            break

        accepted, history, error = nfa.simulate(test_str)
        if error:
            print(f"\n[!] Error: {error}\n")
        else:
            nfa.draw_execution_trace(history, accepted)


if __name__ == "__main__":
    run()

=== Universal NFA Simulator ===
Enter alphabet symbols (e.g. 0 1): 0 1
Enter state names (e.g. q0 q1 q2): q0 q1 q2 q3 q4
Enter start state: q0
Enter accept state(s): q2 q4

Enter transitions. Type destination states separated by space (or press Enter if none):
  δ(q0, 0) -> q0 q1
  δ(q0, 1) -> q0 q3
  δ(q0, ε) -> 
  δ(q1, 0) -> 
  δ(q1, 1) -> q2
  δ(q1, ε) -> 
  δ(q2, 0) -> 
  δ(q2, 1) -> 
  δ(q2, ε) -> 
  δ(q3, 0) -> q4
  δ(q3, 1) -> 
  δ(q3, ε) -> 
  δ(q4, 0) -> 
  δ(q4, 1) -> 
  δ(q4, ε) -> 

                 NFA TRANSITION TABLE
   State     | '0':^8 | '1':^8
------------------------------
-> q0        | {q0,q1}  | {q0,q3} 
   q1        |    ∅     |   {q2}  
   q2 *      |    ∅     |    ∅    
   q3        |   {q4}   |    ∅    
   q4 *      |    ∅     |    ∅    
Legend: '->' = Start State, '*' = Accept State, 'e' = Epsilon (ε)

Enter string to test (or 'exit' to quit): 1110

-------------------------------------------------------
           PARALLEL BRANCH EXECUTION FLOW
-----------

In [ ]:
"""NFA TO DFA CONVERTER (SUBSET CONSTRUCTION)"""
from collections import deque

class NFAToDFAConverter:
    def __init__(self, nfa_states, alphabet, transitions, start_state, accept_states):
        self.nfa_states = set(nfa_states)
        self.alphabet = set(alphabet) - {'e'}  # Alphabet excludes epsilon
        self.nfa_transitions = transitions     # {(state, symbol): {next_states}}
        self.nfa_start = start_state
        self.nfa_accepts = set(accept_states)

    def epsilon_closure(self, states):
        """Finds all NFA states reachable via epsilon (e) transitions from a set of states."""
        stack = list(states)
        closure = set(states)
        while stack:
            state = stack.pop()
            next_states = self.nfa_transitions.get((state, 'e'), set())
            for nxt in next_states:
                if nxt not in closure:
                    closure.add(nxt)
                    stack.append(nxt)
        return frozenset(closure)

    def move(self, states, symbol):
        """Finds all NFA states reachable from a set of states on a given alphabet symbol."""
        result = set()
        for state in states:
            destinations = self.nfa_transitions.get((state, symbol), set())
            result.update(destinations)
        return frozenset(result)

    def convert(self):
        """Performs subset construction to convert NFA to DFA."""
        dfa_transitions = {}
        dfa_accept_states = set()

        # 1. The start state of DFA is the epsilon closure of NFA's start state
        dfa_start = self.epsilon_closure({self.nfa_start})

        unprocessed_states = deque([dfa_start])
        dfa_states = {dfa_start}

        # Mapping to give clean names to DFA subset states (e.g., A, B, C...)
        state_name_map = {dfa_start: "A"}
        counter = 1

        while unprocessed_states:
            current_subset = unprocessed_states.popleft()
            current_name = state_name_map[current_subset]

            # Check if this subset contains any NFA accept state
            if current_subset & self.nfa_accepts:
                dfa_accept_states.add(current_name)

            for symbol in sorted(list(self.alphabet)):
                # Compute transition: closure(move(current_subset, symbol))
                move_result = self.move(current_subset, symbol)
                next_subset = self.epsilon_closure(move_result)

                if not next_subset:
                    # Optional: Handle dead/trap states if needed, or skip empty sets
                    continue

                if next_subset not in dfa_states:
                    dfa_states.add(next_subset)
                    unprocessed_states.append(next_subset)
                    counter += 1
                    state_name_map[next_subset] = chr(ord('A') + (counter - 1) % 26) + (str(counter // 26) if counter > 26 else "")

                next_name = state_name_map[next_subset]
                dfa_transitions[(current_name, symbol)] = next_name

        # Format clean output states dictionary
        formatted_states = sorted(list(state_name_map.values()))
        formatted_accepts = sorted(list(dfa_accept_states))
        formatted_start = state_name_map[dfa_start]

        return formatted_states, sorted(list(self.alphabet)), dfa_transitions, formatted_start, formatted_accepts, state_name_map


def run():
    print("=== NFA to DFA Converter (Subset Construction) ===")
    alphabet = input("Enter NFA alphabet symbols (e.g. 0 1): ").split()
    states = input("Enter NFA state names (e.g. q0 q1 q2): ").split()
    start_state = input("Enter NFA start state: ").strip()
    accept_states = input("Enter NFA accept state(s): ").split()

    print("\nEnter NFA transitions. Type destination states separated by space (or press Enter if none):")
    transitions = {}
    symbols_to_ask = alphabet + ['e']
    for s in states:
        for sym in symbols_to_ask:
            dest = input(f"  δ({s}, {'ε' if sym == 'e' else sym}) -> ").strip()
            if dest:
                transitions[(s, sym)] = set(dest.split())

    converter = NFAToDFAConverter(states, alphabet, transitions, start_state, accept_states)
    dfa_states, dfa_alphabet, dfa_trans, dfa_start, dfa_accepts, mapping = converter.convert()

    print("\n" + "=" * 50)
    print("                 EQUIVALENT DFA RESULT")
    print("=" * 50)
    print(f"DFA States:     {dfa_states}")
    print(f"DFA Alphabet:   {dfa_alphabet}")
    print(f"DFA Start State:{dfa_start}")
    print(f"DFA Accept States: {dfa_accepts}")
    print("\nSubset Mapping (DFA State Name -> NFA State Subsets):")
    for subset, name in sorted(mapping.items(), key=lambda x: x[1]):
        sub_str = "{" + ", ".join(sorted(list(subset))) + "}" if subset else "{∅}"
        print(f"  {name} = {sub_str}")

    print("\nDFA Transition Table:")
    for s in dfa_states:
        for sym in dfa_alphabet:
            dest = dfa_trans.get((s, sym), "TRAP")
            print(f"  δ({s}, {sym}) -> {dest}")
    print("=" * 50)


if __name__ == "__main__":
    run()

=== NFA to DFA Converter (Subset Construction) ===
Enter NFA alphabet symbols (e.g. 0 1): 0 1
Enter NFA state names (e.g. q0 q1 q2): q0 q1 q2 q3
Enter NFA start state: q0
Enter NFA accept state(s): q3

Enter NFA transitions. Type destination states separated by space (or press Enter if none):
  δ(q0, 0) -> q0
  δ(q0, 1) -> q0
  δ(q0, ε) -> q1
  δ(q1, 0) -> q2
  δ(q1, 1) -> 
  δ(q1, ε) -> 
  δ(q2, 0) -> 
  δ(q2, 1) -> q3
  δ(q2, ε) -> 
  δ(q3, 0) -> q3
  δ(q3, 1) -> q3
  δ(q3, ε) -> 

                 EQUIVALENT DFA RESULT
DFA States:     ['A', 'B', 'C', 'D']
DFA Alphabet:   ['0', '1']
DFA Start State:A
DFA Accept States: ['C', 'D']

Subset Mapping (DFA State Name -> NFA State Subsets):
  A = {q0, q1}
  B = {q0, q1, q2}
  C = {q0, q1, q3}
  D = {q0, q1, q2, q3}

DFA Transition Table:
  δ(A, 0) -> B
  δ(A, 1) -> A
  δ(B, 0) -> B
  δ(B, 1) -> C
  δ(C, 0) -> D
  δ(C, 1) -> C
  δ(D, 0) -> D
  δ(D, 1) -> C
